# PHASE 2 V2 — LLM đọc mọi unit + hợp nhất + xuất

Kernel **mới** (CUDA sạch, vLLM là thứ duy nhất chạm CUDA — tránh xung đột
với torch đã init ở Phase 1, xem `PIPELINE_V2.md` §8).

- **K2** — LLM (Qwen3-8B) sinh thực thể cho TỪNG unit trong MỘT lượt
  `generate()` (không còn "vùng mờ" như V1 — mọi unit đều được hỏi)
- **K6** — neo mọi span LLM sinh về văn bản gốc + hàng rào chống ảo giác
  (`span_anchor.verify_unit_entities`) — span không neo được thì BỎ
- **K7** — hợp nhất rule/dict/encoder/llm; heading prior quyết khi 2 nguồn
  xung đột loại trên cùng span (`merge_entities.merge`) — rule/dict luôn
  thắng tuyệt đối
- **K8** — chuẩn biên, mở rộng span cụt bằng KB ICD, xuất đúng format BTC,
  đóng gói zip tải về

**Nguyên lý (NL1):** bỏ sót và trích thừa phạt NGANG NHAU trong WER; chỉ
SAI LOẠI phạt gấp đôi. Nên KHÔNG chặn việc trích (không có ngưỡng "P <
ngưỡng thì bỏ" như V1) — chặn nằm ở K6 (span phải neo được thật) và K7
(rule/dict thắng tuyệt đối khi xung đột loại).

**Cần add Dataset chứa `units/` + `sources/` (output Phase 1).** GPU · Internet ON

In [ ]:
# Cell 1 — env spawn (vLLM là thứ DUY NHẤT chạm CUDA) + cài vllm
import os
os.environ.setdefault('VLLM_WORKER_MULTIPROC_METHOD','spawn')
import sys, glob, json, time, subprocess
subprocess.run([sys.executable,'-m','pip','install','-q','vllm'])
print('vllm cài xong')

In [ ]:
import os
os.environ.setdefault('PYTORCH_CUDA_ALLOC_CONF','expandable_segments:True')
# Cell 2 — dò units/sources + nạp Qwen 1 lần
def find_dir(name):
    h = [x for x in glob.glob(f'/kaggle/input/**/{name}', recursive=True) if os.path.isdir(x)]
    return h[0] if h else (name if os.path.isdir(name) else None)
UNITS = find_dir('units'); SOURCES = find_dir('sources')
assert UNITS and SOURCES, 'thiếu units/ hoặc sources/ — add Dataset output của Phase 1'
if not os.path.exists('fakeer'): subprocess.run(['git','clone','-q','https://github.com/Khanhhh239/fakeer'])
sys.path.insert(0, 'fakeer/src')   # PHẢI trước mọi import từ src/ — bản cũ từng import trước dòng này -> ModuleNotFoundError
os.makedirs('/kaggle/working/final', exist_ok=True)

LLM_MODEL = 'Qwen/Qwen3-8B'   # ≤9B; OOM -> Qwen/Qwen2.5-7B-Instruct
from vllm import LLM, SamplingParams
from transformers import AutoTokenizer
qtok = AutoTokenizer.from_pretrained(LLM_MODEL)
# Qwen3-8B fp16 = 16.4GB > 14.56GB CỦA MỘT T4 -> OOM. Dùng CẢ HAI GPU
# (tensor_parallel_size=2, mỗi con ~8.2GB). max_model_len=3072: prompt V2 có
# system prompt + 5 few-shot (dài hơn hẳn 1 chữ cái của V1) + unit + tối đa
# 160 token sinh ra cho một danh sách thực thể (không phải 1 token nữa).
import torch as _t, inspect as _insp
_ngpu = max(1, _t.cuda.device_count())
print(f'{_ngpu} GPU')

def _mk(model, tp, util):
    want = dict(model=model, dtype='float16', max_model_len=3072,
                gpu_memory_utilization=util, tensor_parallel_size=tp,
                enforce_eager=True, disable_custom_all_reduce=True,
                enable_prefix_caching=True)
    try:
        ok = set(_insp.signature(LLM.__init__).parameters)
        want = {k: v for k, v in want.items() if k in ok}
    except Exception:
        pass
    return LLM(**want)

# Thử dần, GIỮ Qwen3 tới cùng: TP tối đa trước, hạ util, cuối mới đổi model.
_plans = [(LLM_MODEL, _ngpu, 0.90), (LLM_MODEL, _ngpu, 0.80),
          (LLM_MODEL, _ngpu, 0.70), ('Qwen/Qwen2.5-7B-Instruct', _ngpu, 0.85)]
llm = None
for _m, _tp, _u in _plans:
    try:
        print(f'thử {_m} | TP={_tp} | util={_u}')
        llm = _mk(_m, _tp, _u)
        LLM_MODEL = _m
        qtok = AutoTokenizer.from_pretrained(_m)
        break
    except Exception as _e:
        print('  ✗', type(_e).__name__, str(_e)[:110])
assert llm is not None, 'không nạp được LLM nào'
print('DÙNG:', LLM_MODEL)

from span_anchor import verify_unit_entities
from merge_entities import merge
from llm_extract import CODE2TYPE, build_chat_prompts, parse_response
from export_btc import clean_boundary, expand_diagnosis_spans, load_name_set, write_submission, validate
print('modules V2 sẵn sàng')

In [ ]:
# Cell 3 — K2: LLM đọc TỪNG unit của TẤT CẢ file trong MỘT lượt generate()
# (batch càng lớn càng tận dụng tốt bộ lập lịch của vLLM, hơn là gọi generate()
# riêng mỗi file như V1 — và enable_prefix_caching ở Cell 2 khiến phần
# system-prompt + few-shot DÙNG CHUNG cho mọi unit chỉ phải xử lý một lần).
def _fid_key(p):
    b = os.path.splitext(os.path.basename(p))[0]
    return int(b) if b.isdigit() else b
ids = sorted((os.path.splitext(os.path.basename(p))[0] for p in glob.glob(f'{UNITS}/*.json')),
             key=lambda b: int(b) if b.isdigit() else 0)
assert ids, f'không thấy unit nào trong {UNITS}'

all_units, file_range = [], {}
for fid in ids:
    us = json.load(open(f'{UNITS}/{fid}.json', encoding='utf-8'))
    a = len(all_units); all_units.extend(us); file_range[fid] = (a, len(all_units))
print(f'{len(ids)} file | {len(all_units)} unit tổng')

sp = SamplingParams(temperature=0, max_tokens=160)
prompts = build_chat_prompts(all_units, qtok)
t0 = time.time()
outs = llm.generate(prompts, sp)
print(f'LLM sinh xong {len(prompts)} prompt trong {time.time()-t0:.0f}s')
raw_by_unit = [parse_response(o.outputs[0].text) for o in outs]
print(f'tổng đề xuất thô (trước K6): {sum(len(r) for r in raw_by_unit)}')

In [ ]:
# Cell 4 — K6 neo + xác thực từng unit, rồi K7 hợp nhất theo từng file
n_llm_raw = n_llm_kept = 0
for fid in ids:
    a, b = file_range[fid]
    src = json.load(open(f'{SOURCES}/{fid}.json', encoding='utf-8'))
    TEXT = src['text']

    llm_ents = []
    for u, raw in zip(all_units[a:b], raw_by_unit[a:b]):
        n_llm_raw += len(raw)
        llm_ents.extend(verify_unit_entities(raw, u, CODE2TYPE))
    n_llm_kept += len(llm_ents)

    merged = merge(src['rule'], src['dict'], src['encoder'], llm_ents)
    assert all(e['text'] == TEXT[e['start']:e['end']] for e in merged), fid
    o = sorted(merged, key=lambda x: x['start'])
    assert all(o[i]['end'] <= o[i+1]['start'] for i in range(len(o)-1)), f'{fid} chồng lấn'
    json.dump({'text': TEXT, 'entities': o},
              open(f'/kaggle/working/final/{fid}.json', 'w', encoding='utf-8'), ensure_ascii=False)

print(f'\nXONG | LLM đề xuất {n_llm_raw} -> neo được (K6) {n_llm_kept} -> sau hợp nhất (K7) ghi ra final/')
print('=> /kaggle/working/final/*.json')

In [ ]:
# Cell 5 — K8: mở rộng span cụt bằng KB ICD + xuất format BTC + đóng gói zip
_icd = next(iter(glob.glob('/kaggle/input/**/icd10_vi_full.csv', recursive=True)
                 + glob.glob('fakeer/kb/icd10_vi_full.csv')), None)
name_set = load_name_set(_icd) if _icd else None
print(f'KB ICD: {len(name_set) if name_set else 0} tên')

EXPANDED = 0
for fid in ids:
    p = f'/kaggle/working/final/{fid}.json'
    d = json.load(open(p, encoding='utf-8'))
    T = d['text']
    cleaned = []
    for e in d['entities']:
        hit = clean_boundary(T, e['start'], e['end'])
        if not hit:
            continue
        ns, ne = hit
        cleaned.append({**e, 'start': ns, 'end': ne, 'text': T[ns:ne]})
    if name_set:
        EXPANDED += expand_diagnosis_spans(cleaned, T, name_set)
    validate(T, cleaned)
    json.dump({'text': T, 'entities': cleaned}, open(p, 'w', encoding='utf-8'), ensure_ascii=False)
print(f'mở rộng {EXPANDED} span cụt bằng KB ICD')

stats = write_submission('/kaggle/working/final', '/kaggle/working/submit')
print(f"\n{stats['n_files']} file | {stats['n_entities']} thực thể | {stats['by_type']}")
print(f"trung bình {stats['n_entities']/max(1,stats['n_files']):.1f}/file")
assert stats['n_files'] == 100, f"CHỈ CÓ {stats['n_files']} file, phải đủ 100!"

import shutil
shutil.make_archive('/kaggle/working/ner_submit', 'zip', '/kaggle/working/submit')
sz = os.path.getsize('/kaggle/working/ner_submit.zip') / 1024
print(f'\n=> /kaggle/working/ner_submit.zip ({sz:.0f} KB) — tải file này về để nộp')